In [ ]:
%load_ext autoreload
%matplotlib ipympl
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import sys
import clipboard
import numpy as np
import re
import os
from IPython.display import clear_output
sys.path.append('/Users/orenm/BlenderShaderProject/project_files/')

In [ ]:
%autoreload 2
from Logic.blender_tree_manager import BlenderTreeManager
from Logic.tree_networks_manager import TreesNetworkManager

In [ ]:
path = '/Users/orenm/BlenderShaderProject/data/'
images_path = os.path.join(path, 'images/')
db_path = os.path.join(path, 'DB/')

In [ ]:
db_manager = TreesNetworkManager.load(db_path)
len(db_manager.network)

In [ ]:
# read and write string
options = list(db_manager.blender_tree_managers.keys())
strings = []
for i in range(10000):
    if i % 500 == 0:
        print(i)
    some_node = np.random.choice(options)
    btm = db_manager.blender_tree_managers[some_node]
    strings.append(btm.to_str(with_seeds=True))

In [ ]:
with open("nodes_data.txt", "w") as f:
    for line in strings:
        f.write(line + "\n")

In [ ]:
with open("nodes_data.txt", "r") as f:
    strings = [line.strip() for line in f]

# Tokenizer training

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from transformers import AutoTokenizer, PreTrainedTokenizerFast

In [ ]:
# building all the vocab needed for our language
all_strings = set(BlenderTreeManager.NODE_TYPES)
for node in BlenderTreeManager.NODE_TYPES.values():
    all_strings.update(node.NUMERIC)
    cats = node.CATEGORICAL
    for cat in cats.values():
        all_strings.update(cat.options_range)
    all_strings.update(node.CATEGORICAL)
    all_strings.update(node.OUTPUTS)
    all_strings.update(node.SEED)

all_strings = [' '+s for s in all_strings]
special_tokens = ["NODES:", " EDGES:" , " (", " )", " :", " ;", " ->", ' ,', ' [', ' ]', ' |', ' ']
all_strings += special_tokens

numbers = []
for i in range(10):
    numbers.extend([f' {i}', f'_{i}', f'.{i}', f'-{i}', f'{i}'])
all_strings += numbers
all_strings.append('[UNK]')

In [ ]:
# Step 1: Initialize a Byte-Pair Encoding (BPE) tokenizer
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
# Step 2: Pre-tokenization - Ensuring `_1`, `_2` get separated properly
tokenizer.pre_tokenizer = pre_tokenizers.Split(pattern="(_[0-9]+)", behavior="isolated")
# Step 3: Train the tokenizer
trainer = trainers.BpeTrainer(vocab_size=150, min_frequency=10000, special_tokens=all_strings, initial_alphabet=[])
tokenizer.train(["nodes_data.txt"], trainer)

In [ ]:
should_be_all = set(all_strings)
len(should_be_all)

In [ ]:
# verify I don't need the [UNK] token for all my examples
for s in strings:
    encoding = tokenizer.encode(s)
    others = [x for x in encoding.tokens if x not in should_be_all]
    assert len(others) == 0
    assert '[UNK]' not in s

In [ ]:
wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

In [ ]:
wrapped_tokenizer.add_special_tokens({
    "cls_token": "[CLS]",
    "sep_token": "[SEP]",
    "pad_token": "[PAD]",
    "mask_token": "[MASK]"
})

In [ ]:
# should set it to the max in the dataset
wrapped_tokenizer.model_max_length

In [ ]:
wrapped_tokenizer.save_pretrained("./my_tokenizer")